# RISC-V 中国峰会：AME 的 C/C++ 实践教程

从普通数组出发，用 GCC intrinsic 表达一个 **INT32、NN GEMM**，观察 Binutils 生成的机器指令，在修复后的 Spike 上执行，再用独立 Python 参考逐元素校验。本教程固定 **RV64、UDS32/M16/Acc4、硬件 tile N=4**，计算两个行主序 4×4 矩阵的 `C=A×B`。

建议顺序阅读和运行，约需 90–150 分钟；源码重建另行安排。需要 C 数组、指针、函数及基础命令行知识。C++ 使用 C++17，不要求模板经验。章节包括配置与数学模型、C/C++ 核心、执行与排错、动手练习、重建与目标主机验收。

**运行准备：** 从项目根目录打开本 Notebook，选择 Python 3.11+ 内核。源码仓库中的教学文件与本地工具安装分开；运行代码前，将匹配的工具包放置为 `toolchain/` 和 `spike/`。主机需要 x86_64 Linux、Bash、Python、`dtc` 及工具依赖的动态库。Notebook 本身还需 Jupyter 或 IDE 的 Notebook 支持。

“全部运行”会实际编译、执行 Spike 并验证结果，不会启动可选的工具链重建或安装系统软件。生成文件集中在被 Git 忽略的 `build/` 下；每次使用新目录。单位阵练习用临时源码副本，完成后自动删除。单元依次执行，共享 Python 变量；不需要切换 shell 工作目录。

本机已验证环境是 **Ubuntu 24.04.2 LTS / glibc 2.39**。**Ubuntu 26.04.1 LTS / glibc 2.43 尚未实测**，最后一节保留该平台的验收步骤。此处验证功能，不把 Spike 的宿主耗时解释为硬件 GEMM 性能。


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import time

KIT = Path.cwd()
assert (KIT / "scripts/demo.py").is_file(), "请从 ame-tutorial-gcc 项目根目录打开 Notebook"
KIT = KIT.resolve()
BUILD = KIT / "build" / f"notebook-{time.time_ns()}"
BUILD.mkdir(parents=True, exist_ok=False)


def run(argv, *, cwd=KIT, expected_rc=0, timeout=300):
    """显示实际进程输出；精确检查退出码，不把任意失败当作预期负例。"""
    result = subprocess.run([str(arg) for arg in argv], cwd=cwd,
                            capture_output=True, text=True, timeout=timeout)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    if result.returncode != expected_rc:
        raise RuntimeError(f"退出码 {result.returncode}，预期 {expected_rc}: {argv}")
    return result


def read_json(path):
    return json.loads(Path(path).read_text())


def show_source(relative):
    print(f"--- {relative} ---")
    print((KIT / relative).read_text())


print("教学目录:", KIT)
print("本次生成目录:", BUILD)


## 1. 先明确计算和配置

一般 GEMM 中，A 为 `m×k`，B 为 `k×n`，C 为 `m×n`。本例取 `m=n=k=4`，`alpha=1, beta=0`：不转置 A/B，不读取旧 C，计算

\[
C_{ij}=\sum_{k=0}^{3} A_{ik}B_{kj}.
\]

输入采用行主序，第 `(i,j)` 项位于 `base[i*4+j]`。相邻行间隔 4 个 INT32，即 16 字节。本例直接加载行主序 B；数学上按列取 B，并不要求调用者先把 B 转置。

![行主序矩阵中，一行 A 与一列 B 的点积](docs/images/gemm-dot-product.png)

图中每个输出元素来自 A 的一行与 B 的一列。固定输入刻意不对称且带正负数，有助于发现转置、索引与符号错误。

| 概念 | 本例 | 解释 |
| --- | --- | --- |
| XLEN | 64 | 通用整数寄存器宽度；矩阵元素仍为 INT32 |
| 问题维度 m/n/k | 4/4/4 | 算法输入与输出的维度 |
| tile 边长 N | 4 | 由硬件配置及 `amenlen` 观察 |
| Spike 元素数 | 16 | `--ame-nelem=N*N`，这里必须传 16 |
| UDS | 32 | 资源配置的 unit data size，与本例 INT32 基本 tile 匹配 |
| M/Acc 数量 | 16/4 | 矩阵寄存器和累加器资源数，不是矩阵行列数 |

一般 m/n/k 与 N 独立。大矩阵需要分块、尾块处理、布局与累加器设计；不能仅改数组长度或 `--ame-nelem`。

数值范围也属于教程合同：每个输入须为有符号 INT32，且对每个 i、j，`sum(abs(A[i][k]*B[k][j])) <= INT32_MAX`。这个充分条件保证每个乘积和任意部分和均不溢出。本课不覆盖环绕、饱和或宽累加语义。


In [ ]:
inputs = read_json(KIT / "examples/inputs.json")
for name in ("a", "b"):
    print(name.upper())
    for row in inputs[name]:
        print(" ".join(f"{value:5d}" for value in row))
terms = [inputs["a"][0][k] * inputs["b"][k][0] for k in range(4)]
print("C[0][0] 的四项乘积:", terms, "合计:", sum(terms))
assert sum(terms) == -10


### 一个 AME 类型表示一整块矩阵

`__riscv_ztt_i32_rnu_1x1_t` 的 `1x1` 表示一个基本 tile。在 N=4、UDS32 下，它包含 16 个 INT32 元素，不能读成一个标量。`__riscv_ztt_i32_rnu_accx1_t` 是对应的累加器值。

| 对象 | 作用 |
| --- | --- |
| M 值 `av`、`bv` | 保存加载的矩阵操作数 |
| ACC 值 `acc` | 保存乘累加结果，必须先初始化 |
| Md/Ad 状态 | 描述寄存器内容的数据类型，由 intrinsic 展开维护 |
| `rnu` | 整数接口的舍入模式名；本例精确整数运算没有要丢弃的小数位 |

`-mztt-profile=gcc-runtime-u32-m16-a4` 描述编译器可用的 UDS/M/Acc 资源；N 仍由运行平台提供。C runtime 检查 N=4 和 UDS=32，宿主还检查 M16/Acc4。

这些矩阵值属于 opaque 类型，使用带类型的 load/store 与普通数组交换数据；不要假定其内存布局或用普通指针读取内部表示。intrinsic 不保证一个调用只产生一条指令，类型设置、栈保存等都可能产生额外代码。


## 2. 从 C 到 C++：加载、初始化、乘累加、存储

先阅读实际参与编译的核心和 C 入口。函数边界使用普通指针，AME 类型只存在于 kernel 内部。


In [ ]:
show_source("examples/gemm_kernel.h")
show_source("examples/gemm.c")


沿着 `gemm_tile` 读一遍：

1. `zero[16]` 提供零矩阵的普通内存表示。`mls_rm` 按行主序加载 A、B，一个调用加载整个基本 tile；此例不接受额外行 stride。
2. 再加载零 tile，并通过 `mcopy_m2a` 转入 ACC，得到初始 `acc=0`。`mmulacc` 会加上旧累加器，不能省略初始化。
3. `mmulacc_2d(acc, av, bv)` 计算 `acc + av×bv`，矩阵方向为 NN。
4. `mcopy_a2m` 把 ACC 转回 M 值，`mss_rm` 将其写入普通 C 数组。

调用者提供 16 个连续元素的 A、B、C，输出与输入分离。本例 driver 使用 16 字节对齐存储，并通过 `_Static_assert(sizeof(int)==4)` 检查 INT32 假设。

![普通内存、M 矩阵值和 ACC 累加器之间的数据流](docs/images/ame-dataflow.png)

图中箭头对应加载、从零初始化、乘累加及回写。C/C++ API 与汇编命名存在差别：

| intrinsic | ELF 中的关键指令 |
| --- | --- |
| `mls_rm` | `mls.rm` |
| `mcopy_m2a` | `mmov.a.m` |
| `mmulacc_2d` | `mmulacc.2d` |
| `mcopy_a2m` | `mmov.m.a` |
| `mss_rm` | `mss.rm` |

后面会检查最终 ELF，确认实际生成这些指令。寄存器编号、栈帧及额外指令可随优化级别改变。


In [ ]:
show_source("examples/gemm.cpp")


### C++17 和裸机 ABI

`extern "C"` 保留 `tutorial_gemm` 的 C 链接名，让 C driver 可以调用它；函数体仍由 G++ 按 C++17 编译。指针参数遵循 RV64 `lp64` ABI。教程不通过参数、返回值或可变参数传递 AME tile 类型。

本例只使用指针、局部变量和 intrinsic，不调用标准库，不依赖全局构造、异常、RTTI 或 libstdc++。脚本给 C++ 加上 `-fno-exceptions -fno-rtti`。GCC/G++ 及 libgcc 并不意味着已提供完整 C/C++ 运行环境：本包没有 newlib、libstdc++ 或操作系统。

这是 **freestanding** 程序：从 `_start` 开始，自己建立栈、清 BSS 和处理 trap；链接使用 `-nostdlib`。目标端没有 `printf`，终端矩阵由宿主 Python 读取模拟内存后输出。要增加标准 I/O、堆或完整 C++ 标准库，需要另外选择目标运行环境。


### ownership 与本地 CSR 合同

生命周期为：检查 N/UDS 与空闲状态 → `ame.acquire` → 清状态 → 调用 kernel → 检查状态 → `ame.release`。acquire 在普通 C driver 中完成。kernel 返回前所有 AME 结果已存到普通内存，typed M/ACC 值不会跨过 release 存活。

不要随意把 ownership 内联汇编插入有活跃矩阵值的函数。GCC 的 typed ownership 接口另有 region 证明规则，本课不扩展该路径。

| CSR | 本地地址 | 用途 |
| --- | --- | --- |
| `amenlen` / `ameudsz` | `0xCC0` / `0xCC1` | tile 边长与 UDS |
| `ameown` | `0xCC3` | 进入、acquire、release 后的占用状态 |
| `amestatus` | `0x800` | AME 状态 |
| `amestype` | `0x810` | 类型控制 |
| `amefflags` / `amexsat` | `0x811` / `0x812` | 浮点异常与饱和状态；本例保持零 |

这些地址和 `mstatus` 的 AME MS 位属于匹配工具链与修复模型之间的本地合同，不表示正式标准分配。需要配套 GCC、Binutils 和 Spike；仅有相同助记符不能保证二进制兼容。


In [ ]:
show_source("runtime/demo.c")


## 3. 真正编译、反汇编并执行

核心编译选项如下，实际命令会保存到本次生成目录的 `commands.json`：

| 选项 | 作用 |
| --- | --- |
| `-march=rv64im_zicsr_ztt0p6` | 开启本例所需 ISA 与 Ztt v0.6 |
| `-mabi=lp64` | RV64 普通指针/标量 ABI |
| `-mztt-profile=gcc-runtime-u32-m16-a4` | AME 编译资源配置 |
| `-mcmodel=medany` | 配合本例高地址裸机代码布局 |
| `-ffreestanding -fno-builtin` | 使用自带 runtime，避免依赖 hosted 环境 |
| `-nostdlib -static` | 使用本例启动与链接脚本 |
| `-O0` 或 `-O2` | 比较两种优化级别的同一算法 |

先检查实际主机与包内工具。此检查会验证编译器、Binutils、Spike 能否启动，动态库和 `dtc` 是否存在，并确认编译器 helper、AME 头文件与 libgcc 来自当前包。它不会安装软件。


In [ ]:
run([sys.executable, KIT / "scripts/check-host.py", "--output", BUILD / "host.json"])
host = read_json(BUILD / "host.json")
assert host["passed"]
print("目标 26.04.1 / 2.43 是否匹配:", host["requested_target_matched"])


### 第一次执行 C / O2

这一步实际完成 GCC 编译、汇编与链接、Binutils 反汇编、Spike 执行和独立数学参考比较。输出目录必须不存在；重跑这个单元会自动选择新目录。


In [ ]:
c_run = BUILD / f"c-demo-{time.time_ns()}"
run([sys.executable, KIT / "scripts/demo.py", "--language", "c", "--opt", "O2",
     "--out", c_run])
c_case = c_run / "c-O2"
assert read_json(c_case / "report.json")["result"] == "PASS"


### 从源码走到机器指令

| 文件 | 用途 |
| --- | --- |
| `commands.json` | 实际执行的命令 |
| `fixture.h` | 由输入 JSON 生成的目标数组 |
| `gemm.s`、`gemm.elf` | 编译器汇编输出、最终 ELF |
| `disassembly.txt` | 最终 ELF 的反汇编 |
| `spike-debug.cmd` | 本次符号地址对应的调试命令 |
| `observation.json` | 从目标内存和状态读取的结果 |
| `reference.json`、`report.json` | 独立数值比较与该案例结论 |

项目不保留构建日志或文件摘要清单。失败时子进程诊断显示在终端或当前 Notebook 单元；成功后的可读产物放在被忽略的 `build/` 中，便于学习。


In [ ]:
disassembly = (c_case / "disassembly.txt").read_text()
mnemonics = ("mls.rm", "mmov.a.m", "mmulacc.2d", "mmov.m.a", "mss.rm")
for mnemonic in mnemonics:
    assert mnemonic in disassembly, f"最终 ELF 缺少 {mnemonic}"
print("最终 ELF 的 AME 核心指令:")
for line in disassembly.splitlines():
    if any(mnemonic in line for mnemonic in mnemonics):
        print(line)

print("\n实际编译命令（第一项）:")
print(json.dumps(read_json(c_case / "commands.json")[0], indent=2))


### 独立参考如何判断正确

参考程序使用 Python 任意精度整数，按三层标量循环计算 `sum(A[i][k]*B[k][j])`；不调用 AME、Spike helper 或模拟器矩阵实现。它先检查 INT32 输入与无溢出合同，再比较全部 16 个输出。

固定输入预期结果为：

```text
 -10   71   26    38
 -22   27  -28   140
 202  -25  106  -110
  98  213  236    80
```

下面同时查看真实观察、参考结果和参考源代码。A/B 从目标内存读回后也会与 `inputs.json` 比较，避免只用宿主期望输入解释一个执行了其他数据的 ELF。


In [ ]:
observation = read_json(c_case / "observation.json")
reference = read_json(c_case / "reference.json")
assert observation["a"] == inputs["a"] and observation["b"] == inputs["b"]
assert reference["passed"] and reference["checked_elements"] == 16
assert not reference["mismatches"]
print("目标内存 C:")
for row in observation["c"]:
    print(row)
print("独立参考:")
print(json.dumps(reference, indent=2))
show_source("scripts/reference.py")


### 裸机启动、完成 PC 与邮箱

`runtime/spike_boot.S` 放在 `0x80f00000`，关闭中断，启用本模型 provisional AME MS 位（`mstatus` bit25），跳到 `0x80000000` 的 `_start`。`runtime/start.S` 设置 gp/sp/trap 入口并清 BSS，随后调用 C driver。

程序结束时写完成邮箱，执行 `tutorial_done` 中的 `fence rw,rw`，停在 `tutorial_stop` 自循环。宿主在 **fence 之后的完成 PC** 读取内存；目标没有调用系统 `exit`。

| Spike 选项 | 含义 |
| --- | --- |
| `--isa=rv64im_zicsr_ztt --priv=M -p1` | RV64、M-mode、单 hart |
| `-m0x80000000:0x1000000` | 16 MiB RAM |
| `--pc=0x80f00000` | 先执行独立 bootstrap |
| `--ame-nelem=16 --ame-udsz=32` | N=4、UDS32 |
| `--ame-mregs=16 --ame-accregs=4` | M16/Acc4 |

Spike 正常退出只表示调试命令结束。PASS 还要求入口与完成 PC、RV64 M-mode、MS/中断状态、N/UDS/M/Acc、入口和结束 ownership、邮箱正常完成、零状态/flags/XSAT、真实输入、八个边界 canary 与全部 16 个数学结果通过。

`tohost and fromhost symbols not in ELF` 在本例调试器/邮箱协议下正常。此模型 interactive 路径不可靠地扣减 `--instructions` 预算，执行器另外使用 45 秒墙钟超时。


In [ ]:
mailbox = observation["mailbox"]
print("入口 PC:", hex(observation["entry_pc"]))
print("完成 PC:", hex(observation["completion_pc"]))
print("实际 profile:", observation["profile"])
print("mailbox: magic, ABI, stage, error =", mailbox[:4])
assert mailbox[:4] == [0x414D4534, 1, 2, 0]
assert observation["canary_words_checked"] == 8
assert observation["ownership_released"]
for name, lo in (("mcause", 12), ("mepc", 14), ("mtval", 16)):
    print(name, "=", hex(mailbox[lo] | (mailbox[lo + 1] << 32)))


## 4. 实验与自检

### C/C++ × O0/O2，加一个精确负控

下一步实际运行四个正常组合，再运行故意损坏 C[0][0] 的 C/O2 版本。四个正常结果不仅要彼此相同，还要各自与独立参考一致。

负控在 GEMM 完成后把 C[0][0] 的 **32-bit word 异或 1**。固定输入使其从 -10 变成 -9，其余 15 项保持不变。只有程序正常完成、状态与边界检查通过，且独立参考恰好找到 [0,0] 这一处精确 XOR1 差异，才记录 `EXPECTED_REJECTION`。

编译失败、trap、超时或任意其他差异都不算该负控通过。`reference.json` 中的 `passed: false` 表示参考正确拒绝了损坏结果，不能将它混入正常 GEMM 的 PASS。


In [ ]:
all_run = BUILD / f"all-cases-{time.time_ns()}"
run([sys.executable, KIT / "scripts/demo.py", "--language", "all", "--opt", "all",
     "--negative-control", "--out", all_run])
summary = read_json(all_run / "summary.json")
assert summary["passed"] and len(summary["runs"]) == 5
normal_names = ("c-O0", "c-O2", "cpp-O0", "cpp-O2")
for name in normal_names:
    report = read_json(all_run / name / "report.json")
    actual = read_json(all_run / name / "observation.json")["c"]
    assert report["result"] == "PASS"
    assert report["reference"]["passed"] and actual == reference["expected"]
print("四个正常案例分别通过独立参考，并得到同一矩阵。")


In [ ]:
positive = run([sys.executable, KIT / "scripts/reference.py",
                all_run / "c-O2/observation.json"], expected_rc=0)
positive_ref = json.loads(positive.stdout)
assert positive_ref["passed"] and not positive_ref["mismatches"]

negative = run([sys.executable, KIT / "scripts/reference.py",
                all_run / "c-O2-fault/observation.json"], expected_rc=1)
negative_ref = json.loads(negative.stdout)
assert not negative_ref["passed"]
assert len(negative_ref["mismatches"]) == 1
mismatch = negative_ref["mismatches"][0]
assert (mismatch["row"], mismatch["column"]) == (0, 0)
assert (mismatch["actual"] & 0xFFFFFFFF) == ((mismatch["expected"] & 0xFFFFFFFF) ^ 1)
assert mismatch["expected"] == -10 and mismatch["actual"] == -9
assert read_json(all_run / "c-O2-fault/report.json")["result"] == "EXPECTED_REJECTION"
print("正例参考退出 0；负控参考精确退出 1，且只拒绝预期 XOR1 差异。")


### 练习：把 B 换成单位阵

先预测答案：`A×I=A`，所以输出应等于 A。此实验把 `examples/`、`runtime/`、`scripts/` 复制到临时目录，只修改副本中的 B。工具目录使用符号链接供编译和执行读取；练习不写入工具安装，也不重建工具。

这里直接调用 `demo.py`。不对临时副本调用 `check-host.py`，因为主机检查要求 helper 在包内，而本练习有意从外部工具安装读取它们。主包已经在上文完成宿主检查。临时目录退出后自动清理，原始输入保持不变。


In [ ]:
with tempfile.TemporaryDirectory(prefix="ame-identity-", dir=BUILD) as name:
    practice = Path(name)
    for folder in ("examples", "runtime", "scripts"):
        shutil.copytree(KIT / folder, practice / folder,
                        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    for folder in ("toolchain", "spike"):
        (practice / folder).symlink_to(KIT / folder, target_is_directory=True)
    changed_inputs = read_json(practice / "examples/inputs.json")
    changed_inputs["b"] = [[int(i == j) for j in range(4)] for i in range(4)]
    (practice / "examples/inputs.json").write_text(json.dumps(changed_inputs, indent=2) + "\n")
    identity_out = practice / "build/identity"
    run([sys.executable, practice / "scripts/demo.py", "--language", "c", "--opt", "O2",
         "--out", identity_out], cwd=practice)
    identity_observation = read_json(identity_out / "c-O2/observation.json")
    assert identity_observation["c"] == changed_inputs["a"]
    assert read_json(identity_out / "c-O2/reference.json")["passed"]
    print("A × I = A：16 项全部符合预期。")
assert read_json(KIT / "examples/inputs.json") == inputs


### 练习：分别缺少 ISA 和 profile 会怎样

ISA 与资源配置是两个独立维度。第一例保留 profile，把 `-march` 改为 `rv64im_zicsr`；第二例保留 Ztt ISA，移除 `-mztt-profile`。下面实际触发两种编译失败。

| 案例 | 应出现的关键诊断 |
| --- | --- |
| 缺少 ISA 扩展 | `requires the 'ztt0p6' ISA extension` |
| 缺少 profile | `riscv_ztt.h requires a supported -mztt-profile configuration` |

练习脚本要求编译器非零退出且包含对应诊断。“找不到编译器”或无关错误不能当作通过。第二例可能伴随后续未知类型/函数诊断，先修最早的 header/profile 错误。


In [ ]:
diagnostic_out = BUILD / f"compiler-errors-{time.time_ns()}"
run([sys.executable, KIT / "scripts/exercises.py", "--out", diagnostic_out])
diagnostics = read_json(diagnostic_out / "diagnostics.json")
assert {item["name"] for item in diagnostics} == {"missing-isa", "missing-profile"}
for item in diagnostics:
    assert item["passed"] and item["returncode"] != 0
    print(item["name"], "退出码:", item["returncode"])
    print("预期诊断:", item["expected_diagnostic"])


### 出错时从哪一层查

失败诊断显示在当前单元；查看失败目录已经生成的 `commands.json`、汇编和反汇编。尚无 `report.json` 表示完整验证未完成。

| 现象 | 检查顺序 |
| --- | --- |
| 不认识 profile/intrinsic | 包内 GCC 版本和 `riscv_ztt.h` 搜索路径 |
| 汇编器不认识 AME/CSR | GCC 实际调用的 as 是否为匹配 Binutils |
| ELF 缺 AME 指令/属性 | 编译选项、`gemm.s`、`disassembly.txt`、`readelf -hlA` |
| 启动不完整或超时 | bootstrap、装载地址、实际 Spike 参数与报错输出 |
| trap / 邮箱错误 | `mcause`、`mepc`、`mtval`，再看该 PC 附近反汇编 |
| 状态正确但数值不同 | 首差异、目标 A/B、行列布局、零累加器初始化 |

邮箱包含 32 个 u32。word 2 为阶段：1=运行、2=完成、3=失败；word 3 为错误码：1=入口配置/ownership，2=acquire，3=结果状态/release，4=trap。word 12/13、14/15、16/17 分别保存 mcause、mepc、mtval 的低/高 32 位。

Spike 的 `mem` 命令在部分地址返回 64 位，执行器每次读取只取低 u32。若状态检查在生成 `observation.json` 前失败，可根据终端报告的邮箱继续定位；不要把一次 `mem` 输出直接当成完整 64-bit 邮箱字段。

检查工具路径可在终端运行：

```bash
./toolchain/bin/riscv64-unknown-elf-gcc -print-prog-name=as
./toolchain/bin/riscv64-unknown-elf-gcc -print-prog-name=ld
./toolchain/bin/riscv64-unknown-elf-gcc -print-file-name=include
```


### 可选：手动调试同一个 ELF

下面是终端命令说明，不由“全部运行”执行。将 `case_dir` 设为上文打印的某个 `c-O2` 目录。命令向终端输出，不生成日志文件：

```bash
case_dir=/实际路径/build/notebook-.../c-demo-.../c-O2
timeout 45 ./spike/bin/spike \
  --isa=rv64im_zicsr_ztt --priv=M -p1 \
  -m0x80000000:0x1000000 --pc=0x80f00000 \
  --ame-nelem=16 --ame-udsz=32 --ame-mregs=16 --ame-accregs=4 \
  -d "--debug-cmd=$case_dir/spike-debug.cmd" \
  "+payload=$case_dir/spike_boot.elf" "$case_dir/gemm.elf"
```

手工交互时去掉 `--debug-cmd=...`，保留 `-d`。输入 `help` 查看命令；`pc 0`、`reg 0 mstatus`、`ame 0`、`mreg 0 0`、`areg 0 0` 观察状态；`run 1` 单步。`until pc 0 <十六进制地址>` 停到指定 PC，符号地址用 `riscv64-unknown-elf-nm -n` 查询。`quit` 结束。

这只是重放与调试。要产生新的完整校验结论，重新运行 `scripts/demo.py --out 新目录`。


## 5. 从冻结源码重建，再验收另一台主机

此节为可选操作，所有 shell 命令仅展示，不由 Notebook 的“全部运行”执行。首先准备与工具包匹配的完整源码附件，放到 `sources/`：`gcc.tar.gz`、`binutils.tar.gz`、`spike.tar.gz`、`ame-operator.tar.gz`。Git 仓库内的教学代码不能替代这些完整源码。

| 组件 | 本教程使用的来源 |
| --- | --- |
| GCC/G++ | 17.0.0 开发版；AME/Ztt ISA v0.6、intrinsic v0.2.5，本地差异包含于冻结源码 |
| Binutils | 2.47.50.20260930，匹配 Ztt 与本地 CSR |
| Spike | `ead03459` 基线加 14 项修复，冻结于 `e8ecdadf` |
| 目标 | `riscv64-unknown-elf`，演示显式选择 `rv64im_zicsr_ztt0p6/lp64` |

归档已经包含选定修改和 Spike 修复；不要再次把同一修复应用到 `spike.tar.gz`。版本来源见项目 `VERSION.json`。AME-operator 归档用于来源与后续学习，本教程的 GCC/Binutils/Spike 构建不依赖它。

下面使用全新的源码、build 和安装目录，独立于开发工作树，也不复用既有可变构建目录。命令需在同一个终端按顺序执行，以保留变量：

```bash
cd "$HOME/ame-tutorial-gcc"
bundle_dir="$PWD"
rebuild_root=$(mktemp -d "${TMPDIR:-/tmp}/ame-tutorial-rebuild.XXXXXX")
mkdir "$rebuild_root/src"
for component in gcc binutils spike ame-operator; do
  tar -xzpf "$bundle_dir/sources/$component.tar.gz" -C "$rebuild_root/src"
done
```


### 主机构建依赖与 GCC/Binutils

主机需要 C/C++ 编译器、make、GMP/MPFR/MPC、zlib/zstd 开发文件和 `device-tree-compiler`。Spike 使用 C++20，本机采用 G++ 13.3；构建关闭可选 Boost socket 接口，无需为本课安装 Boost。

需要准备 Ubuntu 主机依赖时，由该主机管理员执行：

```bash
sudo apt-get update
sudo apt-get install build-essential bison flex gawk texinfo python3 \
  libgmp-dev libmpfr-dev libmpc-dev zlib1g-dev libzstd-dev \
  device-tree-compiler

df -h "$rebuild_root"
g++ --version
make --version
dtc --version
```

检查可用空间和内存，内存紧张时降低并行数。下面用 `JOBS=4`；`build-toolchain.sh` 接受 1–12：

```bash
SOURCE_ROOT="$rebuild_root/src" \
BUILD_ROOT="$rebuild_root/build/toolchain" \
PREFIX="$rebuild_root/install/toolchain" \
JOBS=4 bash "$bundle_dir/scripts/build-toolchain.sh"
```

脚本先构建并安装 Binutils 的 binutils/gas/ld，再构建 GCC C/C++/LTO 前端和目标 libgcc；拒绝复用非空 build 目录，诊断直接显示在终端。默认基础 ISA/ABI 为 `rv64im_zicsr/lp64`，演示通过明确 `-march` 开启 AME。

`--with-newlib` 是构建配置假设，不表示安装了 newlib。新 prefix 仍是本教程所用 freestanding C/C++ 子集。

检查新编译器：

```bash
"$rebuild_root/install/toolchain/bin/riscv64-unknown-elf-gcc" --version
"$rebuild_root/install/toolchain/bin/riscv64-unknown-elf-g++" --version
"$rebuild_root/install/toolchain/bin/riscv64-unknown-elf-gcc" -print-prog-name=as
"$rebuild_root/install/toolchain/bin/riscv64-unknown-elf-gcc" -print-file-name=include
"$rebuild_root/install/toolchain/bin/riscv64-unknown-elf-gcc" -print-libgcc-file-name
```

相关路径应解析到本次新安装，不能指向开发工程旧工具。


### 重建修复后的 Spike

`build-spike.sh` 接受源码目录、新 build 目录、可选安装 prefix 三个位置参数。显式指定 prefix，让原教学工具保留：

```bash
JOBS=4 bash "$bundle_dir/scripts/build-spike.sh" \
  "$rebuild_root/src/spike" \
  "$rebuild_root/build/spike" \
  "$rebuild_root/install/spike"
```

脚本检查动态库与相对 `$ORIGIN/../lib` RUNPATH，再执行配置/ownership、转换、算术、GEMM、转置累加五个模拟器内部测试。模拟器库静态链接；glibc/libstdc++ 等宿主动态依赖仍由主机提供。内部回归不能代替真实 RISC-V ELF 执行及独立参考。

将少量教学文件放入新安装根，用新 GCC 和新 Spike 验证完整链路：

```bash
mkdir -p "$rebuild_root/install/scripts"
cp "$bundle_dir/scripts/demo.py" "$bundle_dir/scripts/reference.py" \
  "$bundle_dir/scripts/check-host.py" "$bundle_dir/scripts/acceptance.sh" \
  "$rebuild_root/install/scripts/"
cp -a "$bundle_dir/examples" "$bundle_dir/runtime" "$rebuild_root/install/"
bash "$rebuild_root/install/scripts/acceptance.sh" \
  "$rebuild_root/install/build/acceptance"
```

输出应为 C/C++ × O0/O2 四个 PASS、精确数值负控 `EXPECTED_REJECTION`，最后 `ACCEPTANCE_PASS`。输出目录须全新；它记录的是本次新工具实际执行的结论。


### 在 Ubuntu 26.04.1 / glibc 2.43 上验收

把完整工具包放到目标 x86_64 主机，保留目录结构、文件权限及配套源码。运行时需要 Bash、Python 3.11+、coreutils、`ldd`、`dtc` 和相应共享库。

GCC 前端依赖 GMP、MPFR、MPC、zlib、zstd、glibc；Binutils 依赖 zlib、zstd、glibc；Spike 依赖 glibc、libstdc++、libgcc_s、libm。GCC/Binutils 的直接 GLIBC 符号需求最高为 2.38，但静态依赖分析不能替代目标机执行。

在目标主机终端运行：

```bash
cd "$HOME/ame-tutorial-gcc"
target_run="$PWD/build/target-acceptance-$(date +%Y%m%d-%H%M%S)-$$"
env -u GCC_EXEC_PREFIX -u COMPILER_PATH -u LIBRARY_PATH -u LD_LIBRARY_PATH \
  PATH=/usr/bin:/bin \
  bash scripts/acceptance.sh --require-target "$target_run"
```

`--require-target` 检查实际架构为 x86_64、系统 `PRETTY_NAME` 为 `Ubuntu 26.04.1 LTS`、glibc 为 2.43，然后运行完整工具检查与真实演示。如果不匹配就失败。普通本机检查可省略该选项；`requested_target_matched: false` 的本机 PASS 不能被改称为目标平台通过。

目前完成的是 Ubuntu 24.04.2 / glibc 2.39 本机验证。目标环境仍为 **未实测**，只有在该系统完成实际编译、反汇编、Spike 执行和独立参考后，才能增加该平台的实测结论。


## 完成后的自检

- `--ame-nelem=4` 是否代表 N=4？不是，本例必须传 16。
- RV64 是否要求 INT64 矩阵？不要求，XLEN 与矩阵元素类型不同。
- `_1x1_t` 是否表示一个整数？不是，本配置下是一块完整 4×4 tile。
- 为什么不能省略零累加器？`mmulacc` 会加上原有 ACC 值。
- C/C++ 四种组合一致是否足够？还要各自通过独立数学参考与状态检查。
- 为什么负控的参考程序要精确退出 1？它表示有效观察中的数值拒绝，不能用任意失败代替。
- 什么证明程序真的完成？fence 后的完成 PC、邮箱、AME 状态和实际内存共同构成观察。
- 能否把 Spike 的墙钟耗时作为硬件加速比？不能，本教程验证功能。

进一步处理大矩阵，需要设计 m/n/k 分块、尾块零填充、布局与 stride、累加器生命周期和溢出合同。浮点、量化、硬件周期性能及完整 intrinsic API 覆盖属于后续主题。

需要现场 2–3 分钟讲解时，使用项目中的讲者稿与 `scripts/present.sh`；本 Notebook 保留完整自学和实验路径。
